# How to Use ChemSASA with PDB Structures and MD Trajectories

ChemSASA calculates solvent-accessible surface area (SASA) and assigns the atomic SASA values to their corresponding chemical functional groups as defined by the solute partitioning model (SPM).

ChemSASA can be used with either a single PDB structure or an MD trajectory. This notebook provides examples of both using the ubiquitin structure (1UBQ) and a short trajectory from an energy minimization of 1UBQ.

The example input files and the corresponding ChemSASA output files are included in the `example-data` directory. For each calculation, ChemSASA creates a `sasa` directory in the same directory as the input file and writes separate output files containing the atom, residue, chemical functional group, and total SASA.

## Setup

This notebook uses the modular implementation of ChemSASA located in the `chemsasa_modules` directory of the repository. The following cell adds this directory to the Python path so that the ChemSASA modules can be imported directly into the notebook.

Make sure you are running this notebook from the `examples` directory of the cloned ChemSASA repository.

In [1]:
import sys
from pathlib import Path

module_dir = Path("..") / "chemsasa_modules"
sys.path.insert(0, str(module_dir.resolve()))

## Import ChemSASA modules

Import the `SASA` and `UniqueResidues` classes used in this example. `UniqueResidues` already imports the chemical functional-group definitions from `res_func-grp.py`, so they do not need to be imported separately in the notebook.

In [2]:
from SASA import SASA
from UniqueResidues import UniqueResidues

## Calculate SASA for a PDB structure

For a structure calculation, provide the path to the PDB file when creating the `SASA` object.

The `classifier_path` specifies the FreeSASA configuration file containing the atomic radii used for the SASA calculation. Users can provide their own FreeSASA configuration file if they want to use a different set of atomic radii.

`run_structure()` calculates the SASA of the input structure and assigns the atomic SASA values to their corresponding chemical functional groups.

`print_sanity_checks()` checks that MDAnalysis and FreeSASA are reading the structure consistently and that all atoms are accounted for in the functional-group assignment.

`write_output_files()` creates a `sasa` directory containing the atom, residue, chemical functional group, and total SASA output files.

In [ ]:
classifier = "../config-files/freesasa_richards1977.config"
pdb_file = "example-data/1UBQ/1UBQ.pdb"
sasa = SASA(topology=pdb_file, unique_residues=UniqueResidues(), classifier_path=classifier)
sasa.run_structure()
sasa.print_sanity_checks()
sasa.write_output_files()

MDAnalysis atomgroup atom count: 602
MDAnalysis atomgroup residue count: 76
Radii array length: 602
Total mapped atoms (functional groups, from atomgroup): 602
freesasa Structure atom count: 602
freesasa residue count: 76
Total mapped atoms (functional groups, from structure): 602


{'atom': PosixPath('sasa/atom_area_1UBQ.txt'),
 'residue': PosixPath('sasa/residue_area_1UBQ.txt'),
 'func_grp': PosixPath('sasa/func_grp_area_1UBQ.txt'),
 'total': PosixPath('sasa/total_area_1UBQ.txt')}

## Calculate SASA for an MD trajectory

For a trajectory calculation, provide both the topology and trajectory files when creating the `SASA` object. The example below uses a GROMACS `.tpr` topology and `.trr` trajectory.

The `classifier_path` specifies the FreeSASA configuration file containing the atomic radii used for the SASA calculation. Users can provide their own FreeSASA configuration file if they wish to use a different set of atomic radii.

`run_trajectory()` calculates the SASA for each frame of the trajectory 

`print_sanity_checks()` checks the atom and residue counts from the topology, the functional-group assignment, and the number of frames in the trajectory. Structure-based checks are skipped for trajectory calculations.

`write_output_files()` creates a `sasa` directory containing the atom, residue, chemical functional group, and total SASA output files.

In [ ]:
classifier = "../config-files/freesasa_richards1977.config"
topol_file = "example-data/1UBQ-em/topol.tpr"
traj_file = "example-data/1UBQ-em/traj.trr"
sasa = SASA(topology=topol_file, trajectory=traj_file, 
            unique_residues=UniqueResidues(),classifier_path=classifier)
sasa.run_trajectory()
sasa.print_sanity_checks()
sasa.write_output_files()

MDAnalysis atomgroup atom count: 602
MDAnalysis atomgroup residue count: 76
Radii array length: 602
Total mapped atoms (functional groups, from atomgroup): 602
trajectory loaded — structure-based checks skipped
trajectory length/number of frames: 16


{'atom': PosixPath('1UBQ-em/sasa/atom_area_topol.txt'),
 'residue': PosixPath('1UBQ-em/sasa/residue_area_topol.txt'),
 'func_grp': PosixPath('1UBQ-em/sasa/func_grp_area_topol.txt'),
 'total': PosixPath('1UBQ-em/sasa/total_area_topol.txt')}